# Buildify V3 — Architectural LLM Fine-tuning
**Model**: Phi-3.5 Mini (3.8B) | **Method**: QLoRA | **GPU**: Kaggle T4 x2

### Before running — 3 required steps:
1. **Session Options** (right panel) → turn **Internet ON**
2. **Settings** → Accelerator → **GPU T4 x2**
3. Confirm `buildify-training` dataset is attached (right panel → Input)

In [ ]:
# Cell 1 — Install packages (needs Internet ON)
!pip install -q -U \
    transformers \
    datasets \
    peft \
    trl \
    bitsandbytes \
    accelerate \
    sentencepiece

# Verify installs
import importlib
for pkg in ['transformers','datasets','peft','trl','bitsandbytes','accelerate']:
    mod = importlib.import_module(pkg)
    print(f'{pkg}: {mod.__version__}')

In [ ]:
# Cell 2 — Imports + GPU check
import os, json, random, torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer

assert torch.cuda.is_available(), 'No GPU found — enable GPU T4 x2 in Settings → Accelerator'
for i in range(torch.cuda.device_count()):
    gb = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} — {gb:.1f} GB')

In [ ]:
# Cell 3 — Config
MODEL_ID   = 'microsoft/Phi-3.5-mini-instruct'
DATA_FILE  = '/kaggle/input/buildify-training/dataset_clean.jsonl'
SYS_FILE   = '/kaggle/input/buildify-training/architectural_system_prompt.txt'
OUTPUT_DIR = '/kaggle/working/checkpoints'
FINAL_DIR  = '/kaggle/working/final_model'

MAX_SEQ    = 2048   # safe for T4 16GB
BATCH      = 1
GRAD_ACCUM = 16     # effective batch = 16
EPOCHS     = 2
LR         = 2e-4
LORA_R     = 32
LORA_ALPHA = 64
VAL_RATIO  = 0.04

# Verify input files exist
assert Path(DATA_FILE).exists(), f'Dataset not found: {DATA_FILE}'
assert Path(SYS_FILE).exists(),  f'Prompt not found: {SYS_FILE}'

SYS_PROMPT = Path(SYS_FILE).read_text()
print(f'System prompt loaded: {len(SYS_PROMPT)} chars')
print(f'Dataset: {Path(DATA_FILE).stat().st_size / 1e6:.1f} MB')

In [ ]:
# Cell 4 — Load + format dataset
raw = [json.loads(l) for l in Path(DATA_FILE).read_text().splitlines() if l.strip()]
random.seed(42)
random.shuffle(raw)

def fmt(ex):
    return (
        f"<|system|>\n{SYS_PROMPT}<|end|>\n"
        f"<|user|>\n{json.dumps(ex['input'])}<|end|>\n"
        f"<|assistant|>\n{json.dumps(ex['output'], separators=(',', ':'))}<|end|>"
    )

n_val  = max(200, int(len(raw) * VAL_RATIO))
val_ds = Dataset.from_list([{'text': fmt(e)} for e in raw[:n_val]])
trn_ds = Dataset.from_list([{'text': fmt(e)} for e in raw[n_val:]])
print(f'Train: {len(trn_ds):,}   Val: {len(val_ds):,}')

# Sanity check one example
sample = fmt(raw[0])
print(f'Sample length: {len(sample)} chars')

In [ ]:
# Cell 5 — Load model with 4-bit quantization
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print('Model loaded OK')
print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# Cell 6 — Apply LoRA adapters
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias='none',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_up_proj', 'down_proj'
    ],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# Cell 7 — Train
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    evaluation_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    load_best_model_at_end=True,
    logging_steps=20,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    report_to='none',
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=trn_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ,
    packing=True,
)

print('Starting training...')
trainer.train()
print('Training complete!')

In [ ]:
# Cell 8 — Merge LoRA into base model and save
os.makedirs(FINAL_DIR, exist_ok=True)

print('Merging LoRA adapters...')
merged = trainer.model.merge_and_unload()
merged.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f'Model saved to {FINAL_DIR}')

# Zip for easy download
import shutil
print('Zipping...')
shutil.make_archive('/kaggle/working/buildify_arch_llm', 'zip', FINAL_DIR)
size_gb = Path('/kaggle/working/buildify_arch_llm.zip').stat().st_size / 1e9
print(f'Download ready: /kaggle/working/buildify_arch_llm.zip ({size_gb:.1f} GB)')

In [ ]:
# Cell 9 — Smoke test the trained model
test_constraints = {
    'sqft': 2000, 'bedrooms': 3, 'bathrooms': 2, 'stories': 1,
    'style': 'craftsman', 'garage': '2car', 'laundry': 'room',
    'outdoor': 'patio', 'ceilingHeight': 9,
    'primarySuite': True, 'homeOffice': False,
}

prompt = (
    f"<|system|>\n{SYS_PROMPT}<|end|>\n"
    f"<|user|>\n{json.dumps(test_constraints)}<|end|>\n"
    f"<|assistant|>\n"
)

inputs = tokenizer(prompt, return_tensors='pt').to(merged.device)
with torch.no_grad():
    out = merged.generate(
        **inputs,
        max_new_tokens=2000,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

try:
    layout = json.loads(response)
    print(f'PASS — {len(layout["rooms"])} rooms generated')
    print(f'Reasoning: {layout["reasoning"][:150]}')
    print(f'Sqft: {layout["total_conditioned_sqft"]}  Footprint: {layout["footprint_width"]}x{layout["footprint_depth"]}')
    print('\nRooms:')
    for r in layout['rooms']:
        print(f'  {r["name"]:22s} {r["width"]}x{r["height"]} @ ({r["x"]},{r["y"]})')
except json.JSONDecodeError:
    print('JSON parse failed. Raw output:')
    print(response[:800])